# 위험물(라이터 / 보조배터리) 검출 모델 — YOLOX 2-Class (팀 PPE v4 와 같은 포맷)

**런타임 → 런타임 유형 변경 → T4 GPU** 를 고른 뒤 위에서부터 순서대로 실행한다. 학습 약 2~3시간.

| 항목 | 값 |
| :--- | :--- |
| 클래스 | `0 = lighter`, `1 = powerbank` (`normal` 클래스 없음 — 미검출 = NORMAL) |
| 모델 | YOLOX-S (Apache-2.0 · Megvii) — `ppe-v4` 와 같은 계열. ⚠️ ultralytics 는 쓰지 않는다 (AGPL · ADR-24) |
| 입력 | `images` `[1, 3, 640, 640]` float32 · **BGR · 0~255 그대로** · letterbox 좌상단 정렬 · 여백 114 |
| 출력 | `output` `[1, 8400, 7]` — 격자 디코드 전 원시 (오프셋 2 · log 크기 2 · objectness 1 · 클래스 2) |
| 호스트 | `host/vision/detector.py` 의 `YoloxAdapter` 가 그대로 읽는다 (`model_family: yolox`) |

## 준비물 — Google Drive `MyDrive/Hazard_Detection/`
- `lighter_powerbank_coco_2class.zip`

## 결과 — `MyDrive/Hazard_Detection/Hazard_Detection_Model/`
```
weights/   hazard_best.pt · hazard_best.onnx · hazard_best.json(메타데이터) · hazard_exp.py
results/   confusion_matrix.png · results.png · prediction_samples/
metrics/   test_metrics.txt
```
학습 체크포인트는 `MyDrive/Hazard_Detection/yolox_train/` 에 매 epoch 저장된다 (세션이 끊겨도 남는다 → `RESUME = True`).

⚠️ 데이터셋 · 체크포인트는 깃에 올리지 않는다. PPE 모델과 파일 · 데이터 · 코드 모두 분리한다.

## [Cell 1] 설치 — YOLOX 0.3.0 (`tools/ppe/colab_ppe_v5_lowangle.py` 와 같은 방식)

In [ ]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


sh("git clone -q https://github.com/Megvii-BaseDetection/YOLOX /content/YOLOX || true")
sh("cd /content/YOLOX && git checkout -q 0.3.0")
sh("pip -q install loguru tabulate thop ninja pycocotools onnx onnxruntime-gpu")
# ⚠️ `pip install -e .` 은 하지 않는다 — Colab Python 3.13 에서 YOLOX 0.3.0 빌드가 실패한다.
# C++ 확장은 쓰지 않으므로(COCOeval 은 표준판으로 바꾼다) 소스 폴더를 import 경로에 넣기만 한다.
sys.path.insert(0, "/content/YOLOX")
os.environ["PYTHONPATH"] = "/content/YOLOX:" + os.environ.get("PYTHONPATH", "")  # tools/train.py 용
# torch 2.6+ 는 torch.load 기본이 weights_only=True 라 YOLOX 체크포인트를 못 읽는다.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import onnx  # noqa: E402
import onnxruntime  # noqa: E402
import torch  # noqa: E402
from yolox.exp import Exp  # noqa: E402, F401 — 여기서 실패하면 학습 셀까지 가지 않는다

print("YOLOX import OK")
print(
    "torch",
    torch.__version__,
    "| CUDA",
    torch.cuda.is_available(),
    "| onnx",
    onnx.__version__,
    "| onnxruntime",
    onnxruntime.__version__,
)
if not torch.cuda.is_available():
    print("⚠ GPU 미사용 → 런타임 > 런타임 유형 변경 > T4 GPU (CPU 학습은 사실상 불가)")

## [Cell 2] Google Drive Mount

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## [Cell 3] Config — 경로 및 파라미터 (여기만 수정)

In [ ]:
from pathlib import Path

# ===== 경로 =====
DRIVE_DIR = "/content/drive/MyDrive/Hazard_Detection"
DATASET_ZIP = f"{DRIVE_DIR}/lighter_powerbank_coco_2class.zip"
DRIVE_OUT_DIR = f"{DRIVE_DIR}/Hazard_Detection_Model"  # 최종 결과 (PPE 와 분리)
TRAIN_OUT_DIR = f"{DRIVE_DIR}/yolox_train"  # 학습 체크포인트 · 로그 (Drive → 끊겨도 남음)
EXTRACT_DIR = "/content/raw_hazard"  # ZIP 해제 (로컬)
YOLOX_DATA_DIR = "/content/hazard_coco"  # YOLOX 학습 형식 데이터 (로컬)
YOLOX_DIR = "/content/YOLOX"

ANN_FILENAME = "_annotations.coco.json"
SPLITS = ["train", "valid", "test"]

# ===== 클래스 (index = 모델 출력 클래스 번호, 순서 고정) =====
CLASS_NAMES = ("lighter", "powerbank")

# ===== 모델 · 학습 =====
MODEL_SIZE = "s"  # "s" = 팀 ppe-v4 와 같은 YOLOX-S (권장) · "tiny" · "nano" (더 가볍지만 정확도↓)
IMG_SIZE = 640  # ⚠️ 호스트가 640 격자로 디코드한다 — 바꾸면 출력이 [1, 8400, 7] 이 아니다
EPOCHS = 100
NO_AUG_EPOCHS = 15  # 마지막 N epoch 는 mosaic 없이 런타임과 같은 입력으로 마무리
EVAL_INTERVAL = 5  # N epoch 마다 valid 평가 → AP 최고값을 best_ckpt.pth 로 저장
BATCH = 16  # T4 에서 OOM 이면 8
SEED = 20261002
RUN_TRAINING = True  # False = 학습 건너뛰고 기존 best_ckpt.pth 로 평가·export 만
RESUME = False  # True = 끊긴 학습을 latest_ckpt.pth 에서 이어서

# ===== 추론 / 판정 (호스트 vision.* 과 같은 기본값) =====
CONF_THRESHOLD = 0.50  # 위험물 판정 Confidence Threshold
IOU_THRESHOLD = 0.45  # NMS IoU
MATCH_IOU = 0.50  # 평가 시 정답 박스와 같은 물체로 볼 IoU

# ===== 시각화 =====
NUM_VIS_PER_CLASS = 4
NUM_PRED_SAMPLES = 12
NUM_ONNX_SAMPLES = 4

# ===== 메타데이터 (팀 Release · NOTICE 용) — 출처를 확인해서 채운다 =====
MODEL_VERSION = "v1"
DATA_SOURCES = {
    "lighter": "Roboflow Universe vision-wajsm/lighter-wvso3 (CC BY 4.0)",
    "powerbank": "Roboflow Universe vision-wajsm/portable-charger (CC BY 4.0) + ia-act/my-first-project-3xogl (CC BY 4.0)",
    "background": "ia-act/my-first-project-3xogl (CC BY 4.0) + 대상 박스가 없는 vision-wajsm/lighter-wvso3 사진 (CC BY 4.0)",
}

# ===== 파생 경로 (수정 불필요) =====
assert IMG_SIZE == 640, "팀 호스트 포맷은 640 고정"
assert MODEL_SIZE in ("s", "tiny", "nano")
assert CLASS_NAMES == ("lighter", "powerbank") and "normal" not in CLASS_NAMES
EXP_NAME = "hazard_yolox"
EXP_FILE = Path("/content/hazard_exp.py")
PRETRAINED = Path(f"/content/yolox_{MODEL_SIZE}.pth")
RUN_DIR = Path(TRAIN_OUT_DIR) / EXP_NAME
BEST_CKPT = RUN_DIR / "best_ckpt.pth"
LATEST_CKPT = RUN_DIR / "latest_ckpt.pth"
OUT_WEIGHTS = Path(DRIVE_OUT_DIR) / "weights"
OUT_RESULTS = Path(DRIVE_OUT_DIR) / "results"
OUT_PRED = OUT_RESULTS / "prediction_samples"
OUT_METRICS = Path(DRIVE_OUT_DIR) / "metrics"
HAZARD_PT = OUT_WEIGHTS / "hazard_best.pt"
HAZARD_ONNX = OUT_WEIGHTS / "hazard_best.onnx"
HAZARD_META = OUT_WEIGHTS / "hazard_best.json"
METRICS_TXT = OUT_METRICS / "test_metrics.txt"
#: 원본 split → (YOLOX 이미지 폴더, 주석 파일). YOLOX COCODataset 기본 이름이다.
YOLOX_SPLITS = {
    "train": ("train2017", "instances_train.json"),
    "valid": ("val2017", "instances_val.json"),
    "test": ("test2017", "instances_test.json"),
}
for d in (OUT_WEIGHTS, OUT_RESULTS, OUT_PRED, OUT_METRICS, Path(TRAIN_OUT_DIR)):
    d.mkdir(parents=True, exist_ok=True)
print("Config OK →", DRIVE_OUT_DIR)

## [Cell 4] ZIP 압축 해제
지정 경로에 ZIP 이 없으면 내 드라이브 전체에서 같은 파일명을 찾는다.

In [ ]:
import shutil
import zipfile

zp = Path(DATASET_ZIP)
if not zp.exists():
    print(f"지정 경로에 ZIP 없음: {zp}")
    print(f"→ 내 드라이브에서 '{zp.name}' 검색 중...")
    found = sorted(Path("/content/drive/MyDrive").rglob(zp.name))
    print("검색 결과:", [str(f) for f in found] or "없음")
    if found:
        zp = found[0]
        print("→ 사용할 ZIP:", zp, " (Config 의 DATASET_ZIP 도 이 경로로 수정 권장)")

assert zp.exists(), (
    f"ZIP 파일 없음: {DATASET_ZIP}\n"
    "→ ① Colab 에 연결된 Drive 계정 = ZIP 을 올린 계정인지 확인 (!ls /content/drive/MyDrive)\n"
    "→ ② 업로드 직후라면 drive.mount('/content/drive', force_remount=True) 후 재실행"
)
assert zipfile.is_zipfile(zp), f"올바른 ZIP 파일이 아님: {zp}"
DATASET_ZIP_USED = zp

shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
with zipfile.ZipFile(zp) as z:
    bad = z.testzip()
    assert bad is None, f"ZIP 내부 손상 파일: {bad}"
    z.extractall(EXTRACT_DIR)

# _annotations.coco.json 위치로 데이터셋 루트 탐색 (ZIP 안에 상위 폴더가 있어도 동작)
ann_files = [p for p in Path(EXTRACT_DIR).rglob(ANN_FILENAME) if "__MACOSX" not in p.parts]
roots = {p.parent.parent for p in ann_files}
assert len(roots) == 1, f"데이터셋 루트를 하나로 특정할 수 없음: {roots}"
DATA_ROOT = roots.pop()
for s in SPLITS:
    assert (DATA_ROOT / s / ANN_FILENAME).exists(), f"{s}/{ANN_FILENAME} 없음 → ZIP 구조 확인"

print("DATA_ROOT :", DATA_ROOT)
for s in SPLITS:
    print(f"  {s:5s}: {len(list((DATA_ROOT / s).rglob('*'))):5d} files")

## [Cell 5] COCO Annotation 및 데이터셋 구조 검사
split별 이미지 수 · 클래스명 · 클래스별 BBox 수 · Negative 이미지 수 · 누락/손상 이미지 · 비정상 BBox 를 출력한다.
(Annotation 은 수정하지 않고 **검사만** 한다.)

In [ ]:
import json
from collections import Counter, defaultdict

from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def get_img_dir(split):
    """split/images/ 가 있으면 사용, 없으면 split/ 바로 아래 (Roboflow 구조)."""
    d = DATA_ROOT / split
    return d / "images" if (d / "images").is_dir() else d


COCO_SRC = {}
BAD_FILES = {}  # split별 변환 제외 대상 (누락/손상 이미지)
fatal = []

for split in SPLITS:
    with (DATA_ROOT / split / ANN_FILENAME).open(encoding="utf-8") as f:
        coco = json.load(f)
    COCO_SRC[split] = coco
    images, anns = coco["images"], coco["annotations"]
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    img_by_id = {im["id"]: im for im in images}
    img_dir = get_img_dir(split)

    target_ids = {cid for cid, n in cats.items() if n in CLASS_NAMES}
    missing_cls = [n for n in CLASS_NAMES if n not in cats.values()]
    extra_cls = sorted({n for n in cats.values() if n not in CLASS_NAMES})

    box_cnt = Counter(cats.get(a["category_id"], f"UNKNOWN({a['category_id']})") for a in anns)
    img_cls = defaultdict(set)
    for a in anns:
        if a["category_id"] in target_ids:
            img_cls[a["image_id"]].add(cats[a["category_id"]])
    n_neg = sum(1 for im in images if im["id"] not in img_cls)
    n_both = sum(1 for v in img_cls.values() if len(v) == 2)

    # --- 이미지 파일 검사 ---
    missing_files, corrupt, size_mismatch = [], [], []
    for im in images:
        p = img_dir / im["file_name"]
        if not p.exists():
            missing_files.append(im["file_name"])
            continue
        try:
            with Image.open(p) as im_file:
                im_file.verify()
            with Image.open(p) as im_file:
                w, h = im_file.size
            if (w, h) != (im["width"], im["height"]):
                size_mismatch.append((im["file_name"], (w, h), (im["width"], im["height"])))
        except Exception as e:
            corrupt.append((im["file_name"], str(e)))
    json_files = {Path(im["file_name"]).name for im in images}
    disk_files = {p.name for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
    orphan = disk_files - json_files

    # --- BBox 검사 ---
    bad_box = []
    for a in anns:
        im = img_by_id.get(a["image_id"])
        if im is None:
            bad_box.append((a["id"], "image_id 없음"))
            continue
        x, y, w, h = a["bbox"]
        W, H = im["width"], im["height"]
        if w <= 0 or h <= 0:
            bad_box.append((a["id"], f"w/h<=0 {a['bbox']}"))
        elif x < -1 or y < -1 or x + w > W + 1 or y + h > H + 1:
            bad_box.append((a["id"], f"이미지 범위 초과 {a['bbox']} (W={W},H={H})"))

    BAD_FILES[split] = set(missing_files) | {f for f, _ in corrupt}

    print(f"\n========== [{split}] ==========")
    print(f"이미지 수          : {len(images)}  (폴더: {img_dir.relative_to(DATA_ROOT.parent)})")
    print(f"카테고리(JSON)     : {cats}")
    print("클래스별 BBox      : " + ", ".join(f"{k}={v}" for k, v in sorted(box_cnt.items())))
    for n in CLASS_NAMES:
        print(f"  {n:10s} 포함 이미지: {sum(1 for v in img_cls.values() if n in v)}")
    print(f"동시 포함 이미지   : {n_both}")
    print(f"Negative 이미지    : {n_neg}")
    print(f"누락 이미지        : {len(missing_files)} {missing_files[:5]}")
    print(f"손상 이미지        : {len(corrupt)} {corrupt[:5]}")
    print(f"크기 불일치        : {len(size_mismatch)} {size_mismatch[:3]}")
    print(f"JSON에 없는 파일   : {len(orphan)} {sorted(orphan)[:5]}")
    print(f"비정상 BBox        : {len(bad_box)} {bad_box[:5]}")
    if extra_cls:
        print(f"⚠ 학습 제외 클래스 : {extra_cls}  (lighter/powerbank 외 → 변환 시 제외)")
    if missing_cls:
        fatal.append(f"[{split}] 필수 클래스 없음: {missing_cls}")
    if size_mismatch:
        print("⚠ JSON width/height 와 실제 이미지 크기가 다르다 → 박스 위치가 틀어질 수 있음")

assert not fatal, "\n".join(fatal)
print(f"\n검사 완료. 변환 제외 이미지(누락/손상): {sum(len(v) for v in BAD_FILES.values())}장")

## [Cell 6] COCO → YOLOX 학습 형식 정리
YOLOX 는 COCO 주석을 그대로 읽는다 — 좌표 변환은 없고 **폴더 구조와 카테고리 번호만** 맞춘다.
```
hazard_coco/annotations/instances_{train,val,test}.json
hazard_coco/train2017/  val2017/  test2017/
```
- 카테고리는 **이름 기준**으로 `lighter=1`, `powerbank=2` 로 고정 → YOLOX 가 id 순서대로 모델 클래스 `0`, `1` 로 쓴다.
- BBox 값은 원본 그대로 복사한다 (수정 없음). lighter/powerbank 외 클래스는 제외한다.
- Negative 이미지는 박스 없는 이미지로 그대로 넣는다 (`normal` 클래스를 만들지 않는다).

In [ ]:
NAME2CATID = {n: i + 1 for i, n in enumerate(CLASS_NAMES)}

shutil.rmtree(YOLOX_DATA_DIR, ignore_errors=True)
(Path(YOLOX_DATA_DIR) / "annotations").mkdir(parents=True)

CONV_STATS = {}
for split, (img_folder, ann_name) in YOLOX_SPLITS.items():
    coco = COCO_SRC[split]
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    src2dst = {cid: NAME2CATID[n] for cid, n in cats.items() if n in NAME2CATID}
    print(
        f"[{split}] 원본 id → 학습 id : "
        + ", ".join(f"{cid}({cats[cid]})→{v}" for cid, v in src2dst.items())
    )

    names = [Path(im["file_name"]).name for im in coco["images"]]
    dup = [n for n, c in Counter(names).items() if c > 1]
    assert not dup, f"[{split}] 파일명 중복: {dup[:5]}"

    src_dir = get_img_dir(split)
    out_img = Path(YOLOX_DATA_DIR) / img_folder
    out_img.mkdir(parents=True)
    anns_by_img = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_img[a["image_id"]].append(a)

    new = {
        "info": {
            "description": "hazard 2-class (lighter / powerbank) for YOLOX",
            "source": Path(DATASET_ZIP_USED).name,
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {"id": NAME2CATID[n], "name": n, "supercategory": "hazard"} for n in CLASS_NAMES
        ],
    }
    st = Counter()
    for im in coco["images"]:
        if im["file_name"] in BAD_FILES[split]:
            st["skipped_bad_image"] += 1
            continue
        name = Path(im["file_name"]).name
        shutil.copy2(src_dir / im["file_name"], out_img / name)
        img_id = len(new["images"]) + 1
        new["images"].append(
            {"id": img_id, "file_name": name, "width": im["width"], "height": im["height"]}
        )
        n_target = 0
        for a in anns_by_img[im["id"]]:
            if a["category_id"] not in src2dst:
                st["excluded_other_class"] += 1
                continue
            x, y, w, h = a["bbox"]
            if w <= 0 or h <= 0:
                st["skipped_invalid_box"] += 1
                continue
            new["annotations"].append(
                {
                    "id": len(new["annotations"]) + 1,
                    "image_id": img_id,
                    "category_id": src2dst[a["category_id"]],
                    "bbox": [x, y, w, h],
                    "area": a.get("area") or w * h,
                    "iscrowd": a.get("iscrowd", 0),
                }
            )
            st[f"box_{cats[a['category_id']]}"] += 1
            n_target += 1
        st["positive_image" if n_target else "negative_image"] += 1
    (Path(YOLOX_DATA_DIR) / "annotations" / ann_name).write_text(json.dumps(new))
    CONV_STATS[split] = dict(st)

# ----- 검증 ① pycocotools 로 다시 읽기 -----
import contextlib  # noqa: E402
import io  # noqa: E402

from pycocotools.coco import COCO  # noqa: E402

print()
for split, (img_folder, ann_name) in YOLOX_SPLITS.items():
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(Path(YOLOX_DATA_DIR) / "annotations" / ann_name))
    assert sorted(gt.getCatIds()) == [1, 2], gt.getCatIds()
    assert [c["name"] for c in gt.loadCats([1, 2])] == list(CLASS_NAMES)
    for img_id in gt.getImgIds():
        assert (Path(YOLOX_DATA_DIR) / img_folder / gt.loadImgs(img_id)[0]["file_name"]).exists()
    cnt = Counter(gt.loadCats(a["category_id"])[0]["name"] for a in gt.dataset["annotations"])
    st = CONV_STATS[split]
    assert cnt["lighter"] == st.get("box_lighter", 0) and cnt["powerbank"] == st.get(
        "box_powerbank", 0
    )
    print(
        f"[{split}] images={len(gt.getImgIds())}  "
        + "  ".join(f"{k}={v}" for k, v in sorted(st.items()))
    )

# ----- 검증 ② YOLOX 데이터 로더가 보는 클래스 순서 -----
from yolox.data import COCODataset  # noqa: E402

for split, (img_folder, ann_name) in YOLOX_SPLITS.items():
    ds = COCODataset(
        data_dir=YOLOX_DATA_DIR,
        json_file=ann_name,
        name=img_folder,
        img_size=(IMG_SIZE, IMG_SIZE),
        preproc=None,
    )
    assert tuple(ds._classes) == CLASS_NAMES, f"YOLOX 클래스 순서 오류: {ds._classes}"
    idx_cnt = Counter(int(c) for ann in ds.annotations for c in ann[0][:, 4])
    print(f"[{split}] YOLOX 클래스 {ds._classes} → 인덱스별 박스 {dict(sorted(idx_cnt.items()))}")
    assert idx_cnt[0] == CONV_STATS[split].get("box_lighter", 0)
    assert idx_cnt[1] == CONV_STATS[split].get("box_powerbank", 0)
print("\nYOLOX 학습 형식 정리 및 검증 완료 (lighter=0, powerbank=1)")

## [Cell 7] 학습 설정 파일(Exp) 생성 — `tools/ppe/yolox_exp_ppe_s.py` 기준
YOLO 의 `dataset.yaml` 에 해당한다. 증강은 v4 와 같은 방향으로 약하게 둔다:
mosaic 0.5 (배율 0.5~1.5) · mixup 끔 · 회전 ±5° · 좌우반전 0.5 · YOLOX 기본 HSV.

In [ ]:
EXP_FILE.write_text('''
"""YOLOX 위험물 2클래스 (lighter, powerbank) — tools/ppe/yolox_exp_ppe_s.py 기준."""
import os

import torch.nn as nn
import yolox.layers
from pycocotools.cocoeval import COCOeval
from yolox.exp import Exp as BaseExp

# YOLOX 평가기의 C++ COCOeval 즉석 빌드가 실패한다 — 결과가 같은 표준판으로 바꾼다 (팀 v4 와 같음)
yolox.layers.COCOeval_opt = COCOeval

CLASSES = ("lighter", "powerbank")
#: 크기 → (depth, width, depthwise). s 가 팀 ppe-v4 와 같다.
MODEL_SIZES = {"s": (0.33, 0.50, False), "tiny": (0.33, 0.375, False), "nano": (0.33, 0.25, True)}


class Exp(BaseExp):
    def __init__(self):
        super().__init__()
        self.model_size = os.environ.get("HAZARD_MODEL_SIZE", "s")
        self.depth, self.width, self.depthwise = MODEL_SIZES[self.model_size]
        self.num_classes = len(CLASSES)
        self.exp_name = "hazard_yolox"
        self.output_dir = os.environ.get("HAZARD_OUT_DIR", "./YOLOX_outputs")

        self.data_dir = os.environ["HAZARD_DATA_DIR"]
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.test_ann = "instances_test.json"
        # 호스트 어댑터가 640 격자로 디코드한다. 바꾸면 출력이 [1, 8400, 7] 이 아니다.
        self.input_size = self.test_size = (640, 640)
        self.multiscale_range = 2

        # 증강 — 물체 형태를 훼손하지 않는 범위 (v4 와 같은 방향)
        self.mosaic_prob, self.mosaic_scale = 0.5, (0.5, 1.5)
        self.enable_mixup, self.mixup_prob = False, 0.0
        self.hsv_prob = 1.0
        self.flip_prob, self.degrees, self.translate, self.shear = 0.5, 5.0, 0.1, 1.0

        self.max_epoch = int(os.environ.get("HAZARD_EPOCHS", 100))
        self.no_aug_epochs = int(os.environ.get("HAZARD_NO_AUG_EPOCHS", 15))
        self.warmup_epochs = 5
        self.eval_interval = int(os.environ.get("HAZARD_EVAL_INTERVAL", 5))
        self.seed = int(os.environ.get("HAZARD_SEED", 20261002))

    def get_model(self):
        if not self.depthwise:
            return super().get_model()
        # nano 는 depthwise conv — YOLOX 공식 exps/default/yolox_nano.py 와 같다
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model
''')

os.environ.update(
    {
        "HAZARD_DATA_DIR": YOLOX_DATA_DIR,
        "HAZARD_OUT_DIR": TRAIN_OUT_DIR,
        "HAZARD_MODEL_SIZE": MODEL_SIZE,
        "HAZARD_EPOCHS": str(EPOCHS),
        "HAZARD_NO_AUG_EPOCHS": str(NO_AUG_EPOCHS),
        "HAZARD_EVAL_INTERVAL": str(EVAL_INTERVAL),
        "HAZARD_SEED": str(SEED),
    }
)

from yolox.exp import get_exp  # noqa: E402

_exp = get_exp(str(EXP_FILE), None)
assert _exp.num_classes == 2 and _exp.input_size == (640, 640)
print(
    f"Exp OK — YOLOX-{MODEL_SIZE} · depth {_exp.depth} · width {_exp.width} · classes {_exp.num_classes}"
    f" · epochs {_exp.max_epoch} (no-aug {_exp.no_aug_epochs})"
)
del _exp

## [Cell 8] 학습 전 Sample Bounding Box 시각화
**정리된 YOLOX 학습 주석**을 다시 그려 클래스 매핑과 박스가 맞는지 눈으로 확인한다.

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np

COLORS = {0: (0, 140, 255), 1: (255, 200, 0)}  # BGR: lighter=주황, powerbank=하늘


def load_gt(split):
    img_folder, ann_name = YOLOX_SPLITS[split]
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(Path(YOLOX_DATA_DIR) / "annotations" / ann_name))
    return gt, Path(YOLOX_DATA_DIR) / img_folder


def gt_boxes(gt, img_id):
    """[(cls, (x1, y1, x2, y2))] — cls 는 모델 클래스 번호 (0 lighter, 1 powerbank)."""
    out = []
    for a in gt.loadAnns(gt.getAnnIds(imgIds=img_id)):
        x, y, w, h = a["bbox"]
        out.append((a["category_id"] - 1, (x, y, x + w, y + h)))
    return out


def draw_boxes(img, items):
    """items: [(cls, label 문자열, box)]"""
    img = img.copy()
    for c, text, (x1, y1, x2, y2) in items:
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(img, p1, p2, COLORS[c], 3)
        cv2.putText(
            img, text, (p1[0], max(22, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS[c], 2
        )
    return img


random.seed(SEED)
gt_train, train_dir = load_gt("train")
ids = gt_train.getImgIds()
groups = {
    n: [i for i in ids if any(c == k for c, _ in gt_boxes(gt_train, i))]
    for k, n in enumerate(CLASS_NAMES)
}
groups["negative"] = [i for i in ids if not gt_boxes(gt_train, i)]

for gname, gids in groups.items():
    pick = random.sample(gids, min(NUM_VIS_PER_CLASS, len(gids)))
    fig, axes = plt.subplots(1, len(pick), figsize=(4 * len(pick), 4))
    for ax, i in zip(np.atleast_1d(axes), pick, strict=False):
        info = gt_train.loadImgs(i)[0]
        img = cv2.imread(str(train_dir / info["file_name"]))
        vis = draw_boxes(img, [(c, CLASS_NAMES[c], b) for c, b in gt_boxes(gt_train, i)])
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(info["file_name"][:28], fontsize=8)
        ax.axis("off")
    fig.suptitle(f"[train GT] {gname}  (n={len(gids)})", fontsize=13)
    plt.tight_layout()
    plt.show()

## [Cell 9] 모델 학습 — YOLOX `tools/train.py`
- COCO 사전학습 `yolox_s.pth` 에서 시작 → 80클래스 head 는 버리고 **새 2클래스 head 로 학습** (PPE weight 사용 안 함)
- 체크포인트 · 로그는 Drive `yolox_train/` 에 매 epoch 저장. 끊기면 Config 에서 `RESUME = True` 후 이 셀부터 다시 실행
- valid AP 가 가장 높은 epoch 이 `best_ckpt.pth` 로 남는다.
  ⚠️ YOLOX 에는 early stopping(patience) 이 없다 — 대신 best 만 쓰므로 과적합된 마지막 epoch 은 결과에 영향이 없다.

In [ ]:
import re
from collections import deque
from datetime import datetime


def train():
    if RESUME:
        assert LATEST_CKPT.exists(), f"이어 학습할 체크포인트 없음: {LATEST_CKPT}"
        ckpt_arg = f"--resume -c '{LATEST_CKPT}'"
    else:
        if RUN_DIR.exists():  # 이전 결과와 섞이지 않게 옆으로 치운다
            backup = RUN_DIR.with_name(f"{EXP_NAME}_old_{datetime.now():%m%d_%H%M}")
            shutil.move(str(RUN_DIR), str(backup))
            print("이전 학습 폴더 →", backup)
        sh(
            f"wget -q -nc -O {PRETRAINED} "
            f"https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/{PRETRAINED.name}"
        )
        assert PRETRAINED.stat().st_size > 1_000_000, "사전학습 가중치 다운로드 실패"
        ckpt_arg = f"-c '{PRETRAINED}'"
    fp16 = "--fp16" if torch.cuda.is_available() else ""
    cmd = f"cd {YOLOX_DIR} && python tools/train.py -f {EXP_FILE} -d 1 -b {BATCH} {fp16} {ckpt_arg}"
    print("$", cmd)
    # 진행 로그는 epoch 마다 한 줄 + 평가 결과만 보여 준다 (전체는 train_log.txt)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    tail, last_epoch = deque(maxlen=80), None
    for line in proc.stdout:
        tail.append(line)
        m = re.search(r"epoch: (\d+)/(\d+)", line)
        if m and m.group(1) != last_epoch:
            last_epoch = m.group(1)
            print(re.sub(r"^.*? - ", "", line.strip())[:200])
        elif any(
            k in line for k in ("Average Precision", "Save weights", "best", "Error", "Traceback")
        ):
            print(line.rstrip()[:200])
    if proc.wait() != 0:
        print("".join(tail))
        raise RuntimeError("학습 실패 — 위 로그 마지막 부분을 확인")


def plot_training(log_path, out_png):
    """train_log.txt → epoch 별 loss · valid AP 그래프 (results.png)."""
    if not Path(log_path).exists():
        print("⚠ train_log.txt 없음:", log_path)
        return False
    loss = defaultdict(lambda: defaultdict(list))
    ap, ap50 = [], []
    for line in Path(log_path).read_text(errors="ignore").splitlines():
        m = re.search(r"epoch: (\d+)/\d+, iter:", line)
        if m:
            for k, v in re.findall(r"(\w+_loss): ([\d.]+)", line):
                loss[k][int(m.group(1))].append(float(v))
        m = re.search(
            r"Average Precision\s+\(AP\) @\[ IoU=0\.50:0\.95\s*\|\s*area=\s*all\s*\|\s*maxDets=100 \]\s*=\s*([-\d.]+)",
            line,
        )
        if m:
            ap.append(float(m.group(1)))
        m = re.search(
            r"Average Precision\s+\(AP\) @\[ IoU=0\.50\s+\|\s*area=\s*all\s*\|\s*maxDets=100 \]\s*=\s*([-\d.]+)",
            line,
        )
        if m:
            ap50.append(float(m.group(1)))
    if not loss:
        print("⚠ 로그에서 loss 를 찾지 못함 — results.png 생략")
        return False
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for k, per_ep in loss.items():
        eps = sorted(per_ep)
        axes[0].plot(eps, [np.mean(per_ep[e]) for e in eps], label=k)
    axes[0].set(title="train loss (epoch mean)", xlabel="epoch")
    axes[0].legend()
    x = [(i + 1) * EVAL_INTERVAL for i in range(len(ap))]
    axes[1].plot(x, ap50[: len(x)], "o-", label="AP50")
    axes[1].plot(x, ap, "o-", label="AP50:95")
    axes[1].set(title="valid AP", xlabel="epoch (approx.)", ylim=(0, 1))
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(out_png, dpi=120)
    plt.show()
    return True


if RUN_TRAINING:
    train()
else:
    print("RUN_TRAINING = False → 학습 생략")

assert BEST_CKPT.exists(), f"best_ckpt.pth 없음: {BEST_CKPT}"
plot_training(RUN_DIR / "train_log.txt", OUT_RESULTS / "results.png")

# 학습 직후 Drive 에 가중치만 따로 백업 (optimizer 상태 제외 → 용량 절반)
_ck = torch.load(str(BEST_CKPT), map_location="cpu", weights_only=False)
torch.save(
    {
        "model": _ck["model"],
        "start_epoch": _ck.get("start_epoch"),
        "best_ap": _ck.get("best_ap"),
        "classes": list(CLASS_NAMES),
        "model_size": MODEL_SIZE,
        "exp_file": EXP_FILE.read_text(),
    },
    HAZARD_PT,
)
shutil.copy2(EXP_FILE, OUT_WEIGHTS / "hazard_exp.py")
print(f"best epoch {_ck.get('start_epoch')} · valid AP50:95 {_ck.get('best_ap')} → {HAZARD_PT}")
del _ck

## [Cell 10] 추론 공통 함수 — 호스트 `host/vision/detector.py` 의 YoloxAdapter 와 같은 규칙
전처리(BGR · 0~255 · 좌상단 letterbox · 여백 114 · 버림) → 원시 `[1, 8400, 7]` → 격자 디코드 → 클래스별 NMS.
PyTorch 와 ONNX 모두 **같은 함수**로 후처리해서 결과를 비교할 수 있게 한다.

In [ ]:
STRIDES = (8, 16, 32)


def letterbox(img, size=IMG_SIZE):
    """YoloxAdapter.preprocess 와 같다 — BGR · 0~255 · 좌상단 정렬 · 여백 114 · int 버림."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    canvas = np.full((size, size, 3), 114, np.uint8)
    canvas[: int(h * r), : int(w * r)] = cv2.resize(
        img, (int(w * r), int(h * r)), interpolation=cv2.INTER_LINEAR
    )
    return np.ascontiguousarray(canvas.transpose(2, 0, 1)[None], np.float32), r


def _grid(size=IMG_SIZE):
    g, s = [], []
    for st in STRIDES:
        n = size // st
        xv, yv = np.meshgrid(np.arange(n), np.arange(n))
        g.append(np.stack((xv, yv), 2).reshape(-1, 2))
        s.append(np.full((n * n, 1), st))
    return np.concatenate(g), np.concatenate(s)


GRID, GRID_STRIDE = _grid()


def nms(boxes, scores, thr):
    order, keep = scores.argsort()[::-1], []
    areas = np.clip(boxes[:, 2] - boxes[:, 0], 0, None) * np.clip(
        boxes[:, 3] - boxes[:, 1], 0, None
    )
    while order.size:
        b = order[0]
        keep.append(b)
        rest = order[1:]
        iw = np.clip(
            np.minimum(boxes[b, 2], boxes[rest, 2]) - np.maximum(boxes[b, 0], boxes[rest, 0]),
            0,
            None,
        )
        ih = np.clip(
            np.minimum(boxes[b, 3], boxes[rest, 3]) - np.maximum(boxes[b, 1], boxes[rest, 1]),
            0,
            None,
        )
        inter = iw * ih
        union = areas[b] + areas[rest] - inter
        order = rest[np.where(union > 0, inter / np.maximum(union, 1e-9), 0) <= thr]
    return keep


def decode(raw, ratio, img_shape, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD):
    """원시 [1, 8400, 5+C] → [{'cls', 'label', 'score', 'box'(x1,y1,x2,y2 원본 픽셀)}] (점수 내림차순)."""
    raw = np.asarray(raw, np.float32)
    raw = raw[0] if raw.ndim == 3 else raw
    assert raw.shape == (len(GRID), 5 + len(CLASS_NAMES)), f"출력 모양 오류: {raw.shape}"
    with np.errstate(over="ignore"):
        centers = (raw[:, :2] + GRID) * GRID_STRIDE
        half = np.exp(raw[:, 2:4]) * GRID_STRIDE / 2
    boxes = np.concatenate([centers - half, centers + half], 1) / ratio
    cls = raw[:, 5:].argmax(1)
    scores = raw[:, 4] * raw[np.arange(len(raw)), 5 + cls]
    m = scores >= conf
    boxes, scores, cls = boxes[m], scores[m], cls[m]
    h, w = img_shape[:2]
    boxes[:, 0::2] = boxes[:, 0::2].clip(0, w)
    boxes[:, 1::2] = boxes[:, 1::2].clip(0, h)
    out = []
    for k in np.unique(cls):
        mk = cls == k
        for i in nms(boxes[mk], scores[mk], iou):
            out.append(
                {
                    "cls": int(k),
                    "label": CLASS_NAMES[int(k)],
                    "score": float(scores[mk][i]),
                    "box": tuple(float(v) for v in boxes[mk][i]),
                }
            )
    return sorted(out, key=lambda d: -d["score"])


def detect(runner, img_bgr, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD):
    x, r = letterbox(img_bgr)
    return decode(runner(x), r, img_bgr.shape, conf, iou)


def det_items(dets):
    """draw_boxes 입력 형식 — 'lighter 0.91'"""
    return [(d["cls"], f"{d['label']} {d['score']:.2f}", d["box"]) for d in dets]


# ----- PyTorch 모델 (best_ckpt.pth) -----
from torch import nn  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
exp = get_exp(str(EXP_FILE), None)
torch_model = exp.get_model()
_ck = torch.load(str(BEST_CKPT), map_location="cpu", weights_only=False)
torch_model.load_state_dict(_ck["model"])
torch_model.eval().to(DEVICE)
torch_model.head.decode_in_inference = False  # 디코드는 위 decode() 가 한다 (호스트와 같은 구조)
del _ck


def torch_runner(x):
    with torch.no_grad():
        return torch_model(torch.from_numpy(x).to(DEVICE)).float().cpu().numpy()


_probe = torch_runner(np.full((1, 3, IMG_SIZE, IMG_SIZE), 114, np.float32))
assert list(_probe.shape) == [1, 8400, 5 + len(CLASS_NAMES)], _probe.shape
print("PyTorch 원시 출력:", list(_probe.shape), "| device", DEVICE)

## [Cell 11] Test Dataset 평가
- **mAP@0.5 · mAP@0.5:0.95** — pycocotools COCOeval (conf 0.001 전체 후보 · 표준 방식)
- **Precision · Recall** — 운용 기준 `CONF_THRESHOLD` 에서 IoU ≥ `MATCH_IOU` 로 정답과 매칭
- **Confusion Matrix** — 같은 기준, 행 = 예측 / 열 = 정답, background = 미검출(FN) · 오검출(FP)

In [ ]:
import pandas as pd
from pycocotools.cocoeval import COCOeval


def run_split(runner, split="test", conf=0.001):
    """split 전체 추론 → (gt, {image_id: dets})"""
    gt, img_dir = load_gt(split)
    dets = {}
    for img_id in sorted(gt.getImgIds()):
        info = gt.loadImgs(img_id)[0]
        img = cv2.imread(str(img_dir / info["file_name"]))
        assert img is not None, f"이미지 읽기 실패: {info['file_name']}"
        dets[img_id] = detect(runner, img, conf=conf)
    return gt, dets


def box_iou(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def greedy_match(preds, gts, same_class):
    """점수 높은 예측부터 IoU 최대 정답과 짝짓기 → [(pred_idx, gt_idx | None)]"""
    used, pairs = set(), []
    for pi, d in enumerate(preds):
        best, bi = 0.0, None
        for gi, (gc, gb) in enumerate(gts):
            if gi in used or (same_class and gc != d["cls"]):
                continue
            v = box_iou(d["box"], gb)
            if v > best:
                best, bi = v, gi
        if bi is not None and best >= MATCH_IOU:
            used.add(bi)
            pairs.append((pi, bi))
        else:
            pairs.append((pi, None))
    return pairs, used


def evaluate(gt, dets, conf_threshold=CONF_THRESHOLD):
    nc = len(CLASS_NAMES)
    # ① COCO mAP
    results = []
    for img_id, ds in dets.items():
        for d in ds[:100]:
            x1, y1, x2, y2 = d["box"]
            results.append(
                {
                    "image_id": img_id,
                    "category_id": d["cls"] + 1,
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": d["score"],
                }
            )

    def coco_ap(cat_ids):
        if not results:
            return 0.0, 0.0
        with contextlib.redirect_stdout(io.StringIO()):
            ev = COCOeval(gt, gt.loadRes(results), "bbox")
            ev.params.catIds = cat_ids
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
        return max(0.0, float(ev.stats[1])), max(0.0, float(ev.stats[0]))

    # ② Precision / Recall / Confusion Matrix (운용 threshold)
    tp, fp, fn = np.zeros(nc), np.zeros(nc), np.zeros(nc)
    cm = np.zeros((nc + 1, nc + 1), int)
    for img_id, ds in dets.items():
        gts = gt_boxes(gt, img_id)
        preds = [d for d in ds if d["score"] >= conf_threshold]
        pairs, used = greedy_match(preds, gts, same_class=True)
        for pi, gi in pairs:
            (tp if gi is not None else fp)[preds[pi]["cls"]] += 1
        for gi, (gc, _) in enumerate(gts):
            if gi not in used:
                fn[gc] += 1
        pairs, used = greedy_match(preds, gts, same_class=False)
        for pi, gi in pairs:
            cm[preds[pi]["cls"], gts[gi][0] if gi is not None else nc] += 1
        for gi, (gc, _) in enumerate(gts):
            if gi not in used:
                cm[nc, gc] += 1

    metrics = {}
    for c, n in enumerate(CLASS_NAMES):
        ap50, ap = coco_ap([c + 1])
        metrics[n] = {
            "precision": tp[c] / (tp[c] + fp[c]) if tp[c] + fp[c] else 0.0,
            "recall": tp[c] / (tp[c] + fn[c]) if tp[c] + fn[c] else 0.0,
            "map50": ap50,
            "map50_95": ap,
            "tp": int(tp[c]),
            "fp": int(fp[c]),
            "fn": int(fn[c]),
        }
    ap50, ap = coco_ap([c + 1 for c in range(nc)])
    metrics["all"] = {
        "precision": float(np.mean([metrics[n]["precision"] for n in CLASS_NAMES])),
        "recall": float(np.mean([metrics[n]["recall"] for n in CLASS_NAMES])),
        "map50": ap50,
        "map50_95": ap,
    }
    return metrics, cm


def metrics_lines(metrics):
    lines = []
    for key in ("all",) + CLASS_NAMES:
        m = metrics[key]
        extra = f"   (TP {m['tp']} · FP {m['fp']} · FN {m['fn']})" if "tp" in m else ""
        lines += [
            f"[{key.upper() if key == 'all' else key}]{extra}",
            f"  Precision    : {m['precision']:.4f}",
            f"  Recall       : {m['recall']:.4f}",
            f"  mAP@0.5      : {m['map50']:.4f}",
            f"  mAP@0.5:0.95 : {m['map50_95']:.4f}",
        ]
    return lines


def plot_cm(cm, path):
    labels = list(CLASS_NAMES) + ["background"]
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=12,
            )
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set(
        xlabel="True",
        ylabel="Predicted",
        title=f"Confusion Matrix (test · conf>={CONF_THRESHOLD} · IoU>={MATCH_IOU})",
    )
    plt.tight_layout()
    fig.savefig(path, dpi=120)
    plt.show()


gt_test, test_dir = load_gt("test")
_, TEST_DETS = run_split(torch_runner, "test")
TEST_METRICS, TEST_CM = evaluate(gt_test, TEST_DETS)

labels_cm = list(CLASS_NAMES) + ["background"]
cm_df = pd.DataFrame(
    TEST_CM,
    index=[f"pred_{lab}" for lab in labels_cm],
    columns=[f"true_{lab}" for lab in labels_cm],
)
report = "\n".join(
    [
        "=== Hazard Detection Model : Test Metrics ===",
        f"model : {HAZARD_PT.name} (YOLOX-{MODEL_SIZE}, imgsz={IMG_SIZE}, classes={list(CLASS_NAMES)})",
        f"test images : {len(gt_test.getImgIds())}",
        f"Precision/Recall/CM 기준 : conf >= {CONF_THRESHOLD}, NMS IoU {IOU_THRESHOLD}, 매칭 IoU >= {MATCH_IOU}",
        "mAP 기준 : pycocotools COCOeval (conf 0.001)",
        "",
    ]
    + metrics_lines(TEST_METRICS)
    + ["", "[Confusion Matrix]  rows=Predicted, cols=True", cm_df.to_string(), ""]
)
print(report)
METRICS_TXT.write_text(report, encoding="utf-8")
plot_cm(TEST_CM, OUT_RESULTS / "confusion_matrix.png")

## [Cell 12] 실제 검출 결과 확인 (Test 이미지)
임의의 test 이미지에 BBox · Class · Confidence (`lighter 0.91`) 를 그려 출력하고 Drive 에 저장한다.

In [ ]:
random.seed(SEED + 1)
test_ids = sorted(TEST_DETS)
sample_ids = random.sample(test_ids, min(NUM_PRED_SAMPLES, len(test_ids)))

cols = 4
rows = int(np.ceil(len(sample_ids) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, img_id in zip(np.atleast_1d(axes).ravel(), sample_ids, strict=False):
    info = gt_test.loadImgs(img_id)[0]
    img = cv2.imread(str(test_dir / info["file_name"]))
    dets = [d for d in TEST_DETS[img_id] if d["score"] >= CONF_THRESHOLD]
    vis = draw_boxes(img, det_items(dets))
    cv2.imwrite(str(OUT_PRED / f"pt_{info['file_name']}"), vis)
    gt_names = sorted({CLASS_NAMES[c] for c, _ in gt_boxes(gt_test, img_id)}) or ["none"]
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"GT: {', '.join(gt_names)}\n"
        + (", ".join(f"{d['label']} {d['score']:.2f}" for d in dets) or "(no detection)"),
        fontsize=9,
    )
plt.tight_layout()
plt.show()
print("저장 위치:", OUT_PRED)

## [Cell 13] 위험물 상태 판정 Logic
검출 결과 → `NORMAL` / `LIGHTER` / `POWERBANK` / `BOTH`.
입력은 이 노트북의 검출 dict 와 호스트 `host/vision/detector.Detection`(label · score · box) 을 모두 받는다.
test 전체에 대해 **이미지 단위 상태 판정 정확도 · 오경보율 · 미검출률**도 계산한다.

In [ ]:
STATES = ("NORMAL", "LIGHTER", "POWERBANK", "BOTH")


def state_from_labels(labels):
    has_l, has_p = "lighter" in labels, "powerbank" in labels
    if has_l and has_p:
        return "BOTH"
    if has_l:
        return "LIGHTER"
    if has_p:
        return "POWERBANK"
    return "NORMAL"


def get_hazard_state(detections, conf_threshold=CONF_THRESHOLD):
    """한 프레임의 검출 목록 → "NORMAL" / "LIGHTER" / "POWERBANK" / "BOTH"."""
    found = set()
    for d in detections:
        label, score = (d["label"], d["score"]) if isinstance(d, dict) else (d.label, d.score)
        if score >= conf_threshold and label in CLASS_NAMES:
            found.add(label)
    return state_from_labels(found)


def state_eval(gt, dets):
    gt_s = [state_from_labels({CLASS_NAMES[c] for c, _ in gt_boxes(gt, i)}) for i in sorted(dets)]
    pr_s = [get_hazard_state(dets[i]) for i in sorted(dets)]
    return np.array(gt_s), np.array(pr_s)


for img_id in sample_ids:
    print(
        f"{gt_test.loadImgs(img_id)[0]['file_name'][:40]:40s} → {get_hazard_state(TEST_DETS[img_id])}"
    )

GT_STATES, PT_STATES = state_eval(gt_test, TEST_DETS)
state_cm = pd.crosstab(
    pd.Categorical(GT_STATES, STATES),
    pd.Categorical(PT_STATES, STATES),
    rownames=["GT"],
    colnames=["Pred"],
    dropna=False,
)
STATE_ACC = float(np.mean(GT_STATES == PT_STATES))
FALSE_ALARM = (
    float(np.mean(PT_STATES[GT_STATES == "NORMAL"] != "NORMAL"))
    if (GT_STATES == "NORMAL").any()
    else float("nan")
)
MISS_RATE = (
    float(np.mean(PT_STATES[GT_STATES != "NORMAL"] == "NORMAL"))
    if (GT_STATES != "NORMAL").any()
    else float("nan")
)
state_report = "\n".join(
    [
        f"[Image-level Hazard State]  CONF_THRESHOLD={CONF_THRESHOLD}",
        f"  State Accuracy                 : {STATE_ACC:.4f}",
        f"  False Alarm (NORMAL→위험 판정)  : {FALSE_ALARM:.4f}",
        f"  Miss (위험→NORMAL 판정)         : {MISS_RATE:.4f}",
        state_cm.to_string(),
        "",
    ]
)
print("\n" + state_report)
if not (GT_STATES == "BOTH").any():
    print("⚠ test 에 lighter+powerbank 동시 이미지가 없어 BOTH 판정은 검증되지 않았다")
with METRICS_TXT.open("a", encoding="utf-8") as f:
    f.write("\n" + state_report)

## [Cell 14] 모델 저장 + ONNX Export — `tools/ppe/export_ppe.py` 와 같은 규칙
- opset 11 · 입력 `images` `[1, 3, 640, 640]` · 출력 `output` 원시 `[1, 8400, 7]` (`decode_in_inference=False`) · 정적 shape
- PyTorch ↔ ONNX Runtime 원시 출력 최대 오차 ≤ 1e-3 (입력: 0~255 난수 · 상수 114)
- 메타데이터 `hazard_best.json` (팀 `ppe.json` 과 같은 항목)

In [ ]:
import hashlib
import inspect

import onnxruntime as ort
from yolox.models.network_blocks import SiLU
from yolox.utils import replace_module

OPSET, INPUT_NAME, OUTPUT_NAME, PARITY_TOL, PARITY_SEED = 11, "images", "output", 1e-3, 20260928
PREPROCESS = (
    "BGR 순서 · 0~255 float32 그대로(0~1 정규화·평균 보정 없음) · letterbox 좌상단 정렬 · "
    "여백 114 · 리사이즈 크기는 int(버림) · INTER_LINEAR — host/vision/detector.YoloxAdapter"
)
WANT_SHAPE = [1, 8400, 5 + len(CLASS_NAMES)]


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


# 새 Exp 로 모델을 따로 만든다 (Cell 10 모델을 건드리지 않게)
ck = torch.load(str(HAZARD_PT), map_location="cpu", weights_only=False)
export_model = get_exp(str(EXP_FILE), None).get_model()
export_model.load_state_dict(ck["model"])
export_model.eval()
export_model = replace_module(export_model, nn.SiLU, SiLU)
export_model.head.decode_in_inference = False

extra = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}
torch.onnx.export(
    export_model,
    torch.zeros(1, 3, IMG_SIZE, IMG_SIZE),
    str(HAZARD_ONNX),
    input_names=[INPUT_NAME],
    output_names=[OUTPUT_NAME],
    opset_version=OPSET,
    **extra,
)
onnx.checker.check_model(onnx.load(str(HAZARD_ONNX)))

sess_cpu = ort.InferenceSession(str(HAZARD_ONNX), providers=["CPUExecutionProvider"])
rng = np.random.default_rng(PARITY_SEED)
worst, shape = 0.0, []
for x in (
    rng.uniform(0, 255, (1, 3, IMG_SIZE, IMG_SIZE)).astype(np.float32),
    np.full((1, 3, IMG_SIZE, IMG_SIZE), 114, np.float32),
):
    o = sess_cpu.run(None, {INPUT_NAME: x})[0]
    with torch.no_grad():
        t = export_model(torch.from_numpy(x)).numpy()
    shape = list(o.shape)
    worst = max(worst, float(np.abs(o - t).max()))
assert shape == WANT_SHAPE, (
    f"출력 모양 {shape} ≠ {WANT_SHAPE} — decode_in_inference · 입력 크기 확인"
)
assert worst <= PARITY_TOL, f"PyTorch↔ONNX 원시 출력 오차 {worst:.2e} > {PARITY_TOL}"

ONNX_SHA, ONNX_SIZE = sha256_file(HAZARD_ONNX), HAZARD_ONNX.stat().st_size
meta = {
    "name": "hazard",
    "version": MODEL_VERSION,
    "sha256": ONNX_SHA,
    "size_bytes": ONNX_SIZE,
    "family": f"yolox-{MODEL_SIZE}",
    "opset": OPSET,
    "input": {
        "name": INPUT_NAME,
        "shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "dtype": "float32",
        "preprocess": PREPROCESS,
    },
    "output": {
        "name": OUTPUT_NAME,
        "shape": shape,
        "layout": "cx·cy 오프셋 2 · log w·h 2 · objectness(sigmoid) 1 · 클래스(sigmoid) 2 — 격자 디코드 전",
    },
    "classes": list(CLASS_NAMES),
    "trained_at": datetime.fromtimestamp(BEST_CKPT.stat().st_mtime)
    .astimezone()
    .isoformat(timespec="seconds"),
    "checkpoint": {
        "sha256": sha256_file(HAZARD_PT),
        "epoch": ck.get("start_epoch"),
        "best_ap": ck.get("best_ap"),
    },
    "dataset": {
        "zip": Path(DATASET_ZIP_USED).name,
        "zip_sha256": sha256_file(DATASET_ZIP_USED),
        "splits": CONV_STATS,
        "sources": DATA_SOURCES,
        "note": "normal 은 클래스가 아니라 박스 없는 negative 이미지",
    },
    "training": {
        "exp_file": "hazard_exp.py",
        "pretrained": PRETRAINED.name,
        "max_epoch": EPOCHS,
        "no_aug_epochs": NO_AUG_EPOCHS,
        "seed": SEED,
        "batch": BATCH,
        "input_size": [IMG_SIZE, IMG_SIZE],
        "mosaic_prob": 0.5,
        "mosaic_scale": [0.5, 1.5],
        "enable_mixup": False,
        "yolox": "Megvii-BaseDetection/YOLOX 0.3.0",
    },
    "parity": {
        "max_abs_diff": worst,
        "tol": PARITY_TOL,
        "ok": True,
        "inputs": [f"uniform 0~255 (seed {PARITY_SEED})", "상수 114"],
        "runtime": f"onnxruntime {ort.__version__} CPU · torch {torch.__version__}",
    },
    "evaluation": {
        "test": TEST_METRICS,
        "confusion_matrix": TEST_CM.tolist(),
        "state": {"accuracy": STATE_ACC, "false_alarm": FALSE_ALARM, "miss": MISS_RATE},
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
    },
    "host_config": {
        "model_family": "yolox",
        "input_size": IMG_SIZE,
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "classes": list(CLASS_NAMES),
    },
    "status": "XIAO 실측 전 후보",
    "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
}
HAZARD_META.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

for f in (HAZARD_PT, HAZARD_ONNX, HAZARD_META):
    print(f"{f.name:20s} {f.stat().st_size / 1e6:7.2f} MB")
print(f"ONNX 출력 {shape} · parity 최대 오차 {worst:.2e} (≤ {PARITY_TOL}) · sha256 {ONNX_SHA}")
del ck

## [Cell 15] ONNX 단독 추론 검증 (ONNX Runtime · CPU)
`hazard_best.onnx` 만 Load 해서 **test 전체**를 다시 추론하고, PyTorch 결과(mAP · 상태 판정)와 같은지 비교한다.
단순 Export 성공이 아니라 실제 추론 결과가 같은지 확인하는 단계다.

In [ ]:
sess = ort.InferenceSession(str(HAZARD_ONNX), providers=["CPUExecutionProvider"])
inp, out = sess.get_inputs()[0], sess.get_outputs()[0]
print("input :", inp.name, inp.shape, inp.type)
print("output:", out.name, out.shape)
assert inp.name == INPUT_NAME and list(inp.shape) == [1, 3, IMG_SIZE, IMG_SIZE]
assert out.name == OUTPUT_NAME and list(out.shape) == WANT_SHAPE


def onnx_runner(x):
    return sess.run(None, {INPUT_NAME: x})[0]


_, ONNX_DETS = run_split(onnx_runner, "test")
ONNX_METRICS, ONNX_CM = evaluate(gt_test, ONNX_DETS)
_, ONNX_STATES = state_eval(gt_test, ONNX_DETS)

cmp = pd.DataFrame(
    {
        k: {"PyTorch": TEST_METRICS["all"][k], "ONNX": ONNX_METRICS["all"][k]}
        for k in ("precision", "recall", "map50", "map50_95")
    }
).T
cmp["diff"] = (cmp["ONNX"] - cmp["PyTorch"]).abs()
print(cmp.round(4).to_string())
agree = int((ONNX_STATES == PT_STATES).sum())
print(f"\n이미지별 상태 판정 일치: {agree}/{len(PT_STATES)}")
assert cmp["diff"].max() < 0.01, "PyTorch 와 ONNX 성능 차이가 크다 — export 확인"
if agree != len(PT_STATES):
    print("⚠ 일부 불일치 — threshold 경계값의 미세한 수치 차이인지 확인")

# 샘플 시각화 (위험물 이미지 위주 + negative 1장)
pos = [i for i in test_ids if gt_boxes(gt_test, i)]
neg = [i for i in test_ids if not gt_boxes(gt_test, i)]
random.seed(SEED + 2)
onnx_samples = random.sample(pos, min(NUM_ONNX_SAMPLES - 1, len(pos))) + random.sample(
    neg, min(1, len(neg))
)
fig, axes = plt.subplots(1, len(onnx_samples), figsize=(5 * len(onnx_samples), 5))
for ax, img_id in zip(np.atleast_1d(axes), onnx_samples, strict=False):
    info = gt_test.loadImgs(img_id)[0]
    img = cv2.imread(str(test_dir / info["file_name"]))
    dets = detect(onnx_runner, img)  # 실제 운용과 같은 conf
    vis = draw_boxes(img, det_items(dets))
    cv2.imwrite(str(OUT_PRED / f"onnx_{info['file_name']}"), vis)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(
        f"ONNX: {get_hazard_state(dets)}\n"
        + (", ".join(f"{d['label']} {d['score']:.2f}" for d in dets) or "(no detection)"),
        fontsize=9,
    )
plt.tight_layout()
plt.show()

with METRICS_TXT.open("a", encoding="utf-8") as f:
    f.write(
        "\n[ONNX Runtime 검증]\n"
        + cmp.round(4).to_string()
        + f"\n상태 판정 일치 {agree}/{len(PT_STATES)} · parity {worst:.2e}\n"
    )

## [Cell 16] 최종 저장 결과 확인 + 팀 통합용 정보

In [ ]:
required = [HAZARD_PT, HAZARD_ONNX, HAZARD_META, OUT_RESULTS / "confusion_matrix.png", METRICS_TXT]
for f in required + [OUT_RESULTS / "results.png"]:
    print(("✅ " if f.exists() else "❌ ") + str(f.relative_to(DRIVE_OUT_DIR)))
print(f"✅ results/prediction_samples/ ({len(list(OUT_PRED.iterdir()))} files)")

bad = [p for p in Path(DRIVE_OUT_DIR).rglob("*") if "ppe" in p.name.lower()]
assert not bad, f"PPE 관련 파일이 포함되어 있음: {bad}"
assert all(f.exists() for f in required), "필수 결과 파일 누락"

print("\n" + METRICS_TXT.read_text(encoding="utf-8"))
print(f"""
=== 팀 통합용 (팀장 전달) ===
파일 : weights/hazard_best.onnx  ({ONNX_SIZE:,} bytes)
sha256 : {ONNX_SHA}
메타 : weights/hazard_best.json   (클래스 · 입출력 · 전처리 · 평가 · 학습 설정)

config.yaml 예시 (vision 아래):
  hazard:
    model_path: "models/hazard.onnx"
    model_family: yolox
    input_size: {IMG_SIZE}
    conf_threshold: {CONF_THRESHOLD}
    iou_threshold: {IOU_THRESHOLD}
    classes: [lighter, powerbank]
""")